# PCA common-component subtraction ("All-but-the-Top" for steering vectors)

**Question (RUNLOG 030).** Is steering failure caused by the extracted vector being dominated
by a *common component* shared across all concepts? The earlier v_spec probe subtracted the
LOO **supersense** mean (no rescue; RUNLOG 013), but that presumed WordNet's grouping matches
the model's shared structure. RUNLOG 026 showed the dominant shared structure is **global**
(mean |cos| to the all-600 mean = 0.82), not category-shaped. Here we define the common
component with **no WordNet assumption**: the top principal component of the model's own 600
extracted directions (cf. Mu & Viswanath 2018, "All-but-the-Top", who remove the common
mean + top PCs from word embeddings).

**Design.** Llama-3.1-8B Frame4 directions, layer key -17 (depth 15 = the steering-window
peak). PC1 computed from the *uncentered second-moment* matrix of the 600 unit top-1
directions -- this (and projection removal, v - (v.u)u) is invariant to the arbitrary saved
eigenvector signs, so no sign alignment is needed. Each concept's d15 direction has its PC1
projection removed and is renormalized; all other layers copied unchanged (only d15 is
steered). rfmstats copied unchanged: `magn` scaling is the original vector's (approximation,
absorbed by the 4-point coef grid, as in the v_spec builder).

**Steering arm.** `DIRECTIONS_DIR='directions_wordnet_fv4_pca1/'`, `TARGET_KEYS={-17}`,
magn [1,2,3,4], Eval1+Eval6, `RUN_VARIANT='pca1'` -> tag `Lm17_K1ev_magn_fv4_pca1`.
Paired baseline: the K=1 d15 arm (`Lm17_K1ev_magn_fv4`, 268/600).

In [1]:
import csv, io, os, pickle, shutil
import numpy as np
import pandas as pd
import torch

class CPUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        return super().find_class(module, name)

SRC, DST, KEY = 'directions_wordnet_fv4/', 'directions_wordnet_fv4_pca1/', -17
DS = pd.read_csv('bbxdata/bbx_wordnet_ds.csv')
suffix = '_llama_3_8b_it_eng_only'

In [2]:
# --- PC1 of the 600 unit top-1 directions at d15 (uncentered second moment) ---
V = {}
for c in DS.concept:
    M = CPUUnpickler(open(f"{SRC}rfm_{c.replace(' ','_')}{suffix}.pkl", 'rb')).load()[KEY]
    v = (M[0] if M.shape[0] == 3 else M[:, 0]).float()
    V[c] = (v / v.norm()).numpy()
A = np.stack([V[c] for c in DS.concept])          # (600, d), arbitrary signs
S = A.T @ A / len(A)                               # second-moment matrix (sign-invariant)
evals, evecs = np.linalg.eigh(S)
u1 = evecs[:, -1]
var_share = evals[-1] / evals.sum()
proj = A @ u1
print(f"PC1 share of second-moment spectrum: {var_share:.3f}")
print(f"|proj| onto PC1: mean {np.abs(proj).mean():.3f}, q10 {np.quantile(np.abs(proj),.1):.3f}, "
      f"q90 {np.quantile(np.abs(proj),.9):.3f}")
# sanity: PC1 should be ~ the run-026 'global axis'
G = np.sign(A @ A.T.mean(1))  # not needed for subtraction; PC1 alignment check via mean of |cos|

PC1 share of second-moment spectrum: 0.600
|proj| onto PC1: mean 0.769, q10 0.630, q90 0.879


In [3]:
# --- build the subtracted directions dir (d15 row 0 modified; everything else copied) ---
os.makedirs(DST, exist_ok=True)
resid_norms = {}
for c in DS.concept:
    base = f"rfm_{c.replace(' ','_')}{suffix}"
    d = CPUUnpickler(open(f"{SRC}{base}.pkl", 'rb')).load()
    M = d[KEY].clone().float()
    v = M[0] if M.shape[0] == 3 else M[:, 0]
    vn = v / v.norm()
    u = torch.tensor(u1, dtype=vn.dtype)
    resid = vn - (vn @ u) * u
    resid_norms[c] = float(resid.norm())           # how much of the vector survives
    resid = resid / resid.norm()
    if M.shape[0] == 3:
        M[0] = resid
    else:
        M[:, 0] = resid
    d[KEY] = M
    pickle.dump(d, open(f"{DST}{base}.pkl", 'wb'))
    shutil.copy(f"{SRC}{base}_rfmstats.pkl", f"{DST}{base}_rfmstats.pkl")
rn = pd.Series(resid_norms)
print(f"residual norm after PC1 removal: median {rn.median():.3f}, q10 {rn.quantile(.1):.3f}, "
      f"q90 {rn.quantile(.9):.3f}  (small = vector was mostly the common component)")

residual norm after PC1 removal: median 0.622, q10 0.477, q90 0.776  (small = vector was mostly the common component)


## Steering arm

Run (from repo root, after setting `RUN_VARIANT='pca1'`, `DIRECTIONS_DIR='directions_wordnet_fv4_pca1/'`,
`TARGET_KEYS={-17}`, `COEF_BEHAVIOR='magn'`, `FRAME_STYLE='v4'` in `steering_benchmark/run_config.py`):

```
python -m steering_benchmark.eval_generations --model_set llama --model_size 8B --concepts_to_steer wordnet
python -m steering_benchmark.parse_results    --model_set llama --model_size 8B --concepts_to_steer wordnet
```

In [4]:
# --- results: paired comparison vs the K=1 d15 baseline (run after the steering arm) ---
from scipy.stats import binomtest

def load(tag):
    u = {}
    for vl in ('', '_v6'):
        f = f"csvs/rfm_wordnet_gpt_oss_outputs_500_concepts_llama_3.1_8B_english_only{vl}_{tag}.csv"
        if not os.path.exists(f):
            return None
        for r in list(csv.reader(open(f)))[1:]:
            u[r[0]] = u.get(r[0], False) or float(r[1]) >= 0.5
    return u

base = load('Lm17_K1ev_magn_fv4')
pca = load('Lm17_K1ev_magn_fv4_pca1')
if pca is None:
    print('steering arm not run/judged yet')
else:
    df = DS.copy()
    df['base'] = df.concept.map(base); df['pca'] = df.concept.map(pca)
    df['resid'] = df.concept.map(resid_norms)
    g = df[~df.base & df.pca]; l = df[df.base & ~df.pca]
    p = binomtest(len(g), len(g) + len(l)).pvalue
    print(f"baseline {df.base.sum()}/600  |  PC1-subtracted {df.pca.sum()}/600")
    print(f"gained {len(g)} / lost {len(l)}  McNemar exact p={p:.4f}")
    print(f"residual-norm: gained median {g.resid.median():.3f}, lost {l.resid.median():.3f}, all {df.resid.median():.3f}")
    print('\ngained:', ', '.join(sorted(g.concept)))
    print('\nlost:', ', '.join(sorted(l.concept)))
    print('\nby supersense (base -> pca):')
    s = df.groupby('supersense')[['base','pca']].sum()
    s['diff'] = s.pca - s.base
    print(s.sort_values('diff').to_string())

baseline 268/600  |  PC1-subtracted 234/600
gained 42 / lost 76  McNemar exact p=0.0022
residual-norm: gained median 0.568, lost 0.622, all 0.622

gained: aid, albacore, allowance, candlepower, carp, circle, concession, consulate, cornerstone, dandelion, digit, dogsled, east, eclipse, glycerin, gymnasium, heredity, hickory, latrine, lentil, manhole, marmalade, menstruation, millisecond, nucleus, oxidation, pancake, panhandle, plasma, proctology, propensity, protectiveness, pyramid, rebate, ricotta, shingle, slush, subcompact, synchronization, thermal, turpentine, wetland

lost: accommodation, alarm, backseat, blaze, bounty, brilliance, childhood, coma, control, country, cove, crowd, crunch, darling, decay, earth, electron, emersion, eroding, exhalation, fanaticism, flaming, fluff, gangrene, headphone, headway, hitter, hostage, hurricane, hush, immersion, inertia, innards, institution, invisibility, lift, masochist, meow, midair, mixer, neglect, noonday, octopus, ozone, parade, pectoral